In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os,sys
#sys.path.append('/work/qdiff/mo_utils')
sys.executable


'/home/nadavg/anaconda3/envs/qdiff/bin/python'

In [4]:
print(os.getcwdb())
os.chdir('/home/nadavg/q-diffusion')
print(os.getcwdb())

b'/home/nadavg/q-diffusion/scripts/hf15'
b'/home/nadavg/q-diffusion'


In [5]:
os.environ['CUDA_VISIBLE_DEVICES'] = '4'

In [6]:
from mo_utils.utils.stand_alone_utils.har_utils import get_har_files,get_params_from_har
from mo_utils.utils.stand_alone_utils.pytorch2accelras import get_nested_attr,get_weight_and_bias_from_layer_name
from mo_utils.utils.stand_alone_utils.quant_utils import calc_snr


libtmux not installed ??


In [7]:
from src.utils.torch_utils import add_full_name_to_module
from scripts.hf15.init_pipe import init_pipe

In [8]:
import netron
from pathlib import Path

In [172]:
import torch
import torch.nn as nn
from diffusers import StableDiffusionPipeline

In [10]:
pipe = init_pipe()

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [10]:
#pipe.to('cuda')

In [11]:
unet = pipe.unet
add_full_name_to_module(unet)

In [12]:
types_modules = dict()
for module in unet.modules():
    if type(module) not in types_modules:
        types_modules[type(module)] = module.full_name
    


In [13]:
#types_modules

In [14]:
ace_unet_path =  '/genai/users/ellaf/stable_diffusion/unet/shared_unet_demo/unet_sim_fp.har'

In [15]:
get_har_files(ace_unet_path)

['unet_sim.hn',
 'unet_sim.alls',
 'unet_sim.native.hn',
 'unet_sim.npz',
 'unet_sim.fpo.npz',
 'unet_sim.stats.npz',
 'unet_sim.original_model_meta.json',
 'unet_sim.metadata.json']

In [157]:
params = get_params_from_har(ace_unet_path,params_name='unet_sim.fpo.npz')
hn = get_params_from_har(ace_unet_path,params_name='unet_sim.hn')

all_names=['unet_sim.hn', 'unet_sim.alls', 'unet_sim.native.hn', 'unet_sim.npz', 'unet_sim.fpo.npz', 'unet_sim.stats.npz', 'unet_sim.original_model_meta.json', 'unet_sim.metadata.json']
loading  params_name='unet_sim.fpo.npz' ...
all_names=['unet_sim.hn', 'unet_sim.alls', 'unet_sim.native.hn', 'unet_sim.npz', 'unet_sim.fpo.npz', 'unet_sim.stats.npz', 'unet_sim.original_model_meta.json', 'unet_sim.metadata.json']
loading  params_name='unet_sim.hn' ...


In [29]:
type(hn['layers'])

dict

In [32]:
#ii=iter(hn['layers'])
layer

{'type': 'input_layer',
 'input': [],
 'output': ['unet_sim/conv1'],
 'input_shapes': [[-1, 64, 64, 4]],
 'output_shapes': [[-1, 64, 64, 4]],
 'original_names': ['sample'],
 'compilation_params': {},
 'quantization_params': {},
 'transposed': False,
 'engine': 'nn_core',
 'io_type': 'standard'}

In [33]:
types_layers = dict()
for layer_name , layer in hn['layers'].items():
    if layer['type'] not in types_layers:
        types_layers[layer['type']] = layer_name

In [34]:
types_layers

{'input_layer': 'unet_sim/input_layer1',
 'conv': 'unet_sim/conv1',
 'dense': 'unet_sim/fc1',
 'layer_normalization': 'unet_sim/layer_normalization1',
 'normalization': 'unet_sim/normalization1',
 'resize': 'unet_sim/resize1',
 'ew_add': 'unet_sim/ew_add1',
 'format_conversion': 'unet_sim/format_conversion1',
 'matmul': 'unet_sim/matmul1',
 'reduce_max': 'unet_sim/reduce_max_softmax1',
 'ew_sub': 'unet_sim/ew_sub_softmax1',
 'reduce_sum': 'unet_sim/reduce_sum_softmax1',
 'ew_mult': 'unet_sim/ew_mult_softmax1',
 'concat': 'unet_sim/concat1',
 'output_layer': 'unet_sim/output_layer1'}

In [158]:
layers_names = list(hn['layers'].keys())
layers_names[:10]

['unet_sim/input_layer1',
 'unet_sim/conv1',
 'unet_sim/input_layer2',
 'unet_sim/fc1',
 'unet_sim/fc2',
 'unet_sim/fc10',
 'unet_sim/fc11',
 'unet_sim/fc12',
 'unet_sim/fc13',
 'unet_sim/fc14']

In [18]:
#get_nested_attr,
ind =73
ker,bais,kk,bk=get_weight_and_bias_from_layer_name(layers_names[ind],params)
if ker is not None:
    print(ker.shape)
if bais is not None:
    print(bais.shape)
print(kk,bk)
print(f'lname={layers_names[ind]} type={hn["layers"][layers_names[ind]]["type"]}')
org_names =hn['layers'][layers_names[ind]]['original_names']
print(f'{org_names=}')

(1, 1, 320, 1)
(320,)
unet_sim/normalization2/kernel:0 unet_sim/normalization2/bias:0
lname=unet_sim/normalization2 type=normalization
org_names=['/down_blocks.0/resnets.0/norm2/Mul', '/down_blocks.0/resnets.0/norm2/Add', '/down_blocks.0/resnets.0/act_2/Sigmoid', '/down_blocks.0/resnets.0/act_2/Mul']


In [ ]:
# 1. if type = layer_normalization  - do nothing
# 2. if type = normalization -> find the layer normalization layer and update weights and bias
# 1. if LayerNormalization in org_names[0] and len(org_names)>1:
#  norm are inside conv values.

In [ ]:
def get_relevent_orig_name(org_names,):
    for org_name in org_names:
        if 'LayerNormalization' in org_name:
            return org_name
    return None

In [50]:
get_nested_attr(unet,layers_names[ind])

AttributeError: 'UNet2DConditionModel' object has no attribute 'unet_sim'

In [154]:
layers_names[ind]

'unet_sim/conv_feature_splitter13_3'

In [155]:
params.keys()

dict_keys(['unet_sim/conv1/padding_const_value:0', 'unet_sim/conv1/kernel:0', 'unet_sim/conv1/bias:0', 'unet_sim/layer_normalization103/epsilon:0', 'unet_sim/normalization103/kernel:0', 'unet_sim/normalization103/bias:0', 'unet_sim/layer_normalization1/epsilon:0', 'unet_sim/normalization1/kernel:0', 'unet_sim/normalization1/bias:0', 'unet_sim/fc1/kernel:0', 'unet_sim/fc1/bias:0', 'unet_sim/conv12/padding_const_value:0', 'unet_sim/conv12/kernel:0', 'unet_sim/conv12/bias:0', 'unet_sim/mul_and_add1/kernel:0', 'unet_sim/mul_and_add1/bias:0', 'unet_sim/conv23/padding_const_value:0', 'unet_sim/conv23/kernel:0', 'unet_sim/conv23/bias:0', 'unet_sim/conv28/padding_const_value:0', 'unet_sim/conv28/kernel:0', 'unet_sim/conv28/bias:0', 'unet_sim/mul_and_add9/kernel:0', 'unet_sim/mul_and_add9/bias:0', 'unet_sim/conv29/padding_const_value:0', 'unet_sim/conv29/kernel:0', 'unet_sim/conv29/bias:0', 'unet_sim/conv30/padding_const_value:0', 'unet_sim/conv30/kernel:0', 'unet_sim/conv30/bias:0', 'unet_sim/

In [153]:
[k for k in params.keys() if layers_names[ind] in k]

[]

In [240]:
# update conv layer. 
ind =608 #362#357
ker,bias,kk,bk=get_weight_and_bias_from_layer_name(layers_names[ind],params)
if ker is not None:
    print(ker.shape)
if bias is not None:
    print(bias.shape)
print(kk,bk)
print(f'lname={layers_names[ind]} type={hn["layers"][layers_names[ind]]["type"]}')
org_names =hn['layers'][layers_names[ind]]['original_names']
print(f'{org_names=}')


(1, 1, 640, 2560)
(2560,)
unet_sim/conv_feature_splitter22_2/kernel:0 unet_sim/conv_feature_splitter22_2/bias:0
lname=unet_sim/conv_feature_splitter22_2 type=conv
org_names=['/up_blocks.2/attentions.0/transformer_blocks.0/norm3/LayerNormalization', '/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/proj/MatMul', '/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/proj/Add', '/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/Slice', '/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/Slice_1', '/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/Div_1', '/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/Erf', '/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/Add_1', '/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/Mul_2', '/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/Mul_3']


In [199]:
layer_norm_input = False
ff_input = False
kqv = False
if any('layer_normalization' in s for s in hn['layers'][layers_names[ind]]['input']):
    layer_norm_input = True
    if 'conv_feature_splitter' in layers_names[ind]:
        kqv = True
if any (['ff' in on for on  in org_names]):
    ff_input = True
print(f'{layer_norm_input=},{ff_input=},{kqv=}')
    

layer_norm_input=True,ff_input=False,kqv=False


In [162]:
org_names

['/mid_block/attentions.0/transformer_blocks.0/norm1/LayerNormalization',
 '_v_4950',
 '_v_4951',
 '/mid_block/attentions.0/transformer_blocks.0/attn1/Reshape',
 '/mid_block/attentions.0/transformer_blocks.0/attn1/Reshape_1',
 '/mid_block/attentions.0/transformer_blocks.0/attn1/Reshape_2',
 '/mid_block/attentions.0/transformer_blocks.0/attn1/Transpose',
 '/mid_block/attentions.0/transformer_blocks.0/attn1/Transpose_2',
 '/mid_block/attentions.0/transformer_blocks.0/attn1/Transpose_1']

In [164]:
if layer_norm_input is False and ff_input is False:
    module = get_nested_attr(unet,org_names[0])
    p_weights = module.weight.data#.shape,
    p_bias = module.bias.data#.shape
    #p_weights.shape,p_weights.shape
    assert(p_weights.numel() == ker.size)
    assert(p_bias.numel() == bias.size)
    p_ker = acc_ker_to_pytorch_weight(ker)
    assert(p_weights.shape == p_ker.shape)
    module.weight.data =  p_ker
    module.bias.data =  torch.from_numpy(bias).reshape(p_bias.shape)

In [197]:
if layer_norm_input and kqv :
    if layers_names[ind].endswith('_1'):
        suffix = 'to_k'
    elif layers_names[ind].endswith('_2'):
        suffix = 'to_q'
    elif layers_names[ind].endswith('_3'):
        suffix = 'to_v'
    else:
        raise ValueError(f'layer name {layers_names[ind]} should end with _1 or _2 or _3')
    
    org_name_=None
    for org_name in org_names:
        if 'attn' in org_name:
            org_name_ = org_name
            break
    assert(org_name_ is not None)
    attn_org_name = org_name_.rpartition("/")[0] 
    org_name = attn_org_name+ '/' + suffix
    module = get_nested_attr(unet,org_name,ignore_last=0)
    p_weights = module.weight.data#.shape,
    if module.bias is not None:
        p_bias = module.bias.data
    else:
        p_bias = None
    assert(p_weights.numel() == ker.size)
    if p_bias is not None:
        assert(p_bias.numel() == bias.size)
    p_ker = acc_ker_to_pytorch_weight(ker)
    assert(p_weights.shape == p_ker.shape)
    module.weight.data =  p_ker
    if bias is not None:
        if p_bias is not None:
            module.bias.data =  torch.from_numpy(bias).reshape(p_bias.shape)
        else:
            module.bias =  nn.Parameter(torch.from_numpy(bias).reshape(-1))
    else:
        module.bias = None
    
    ## remove weight and bias from layer_normalization. 
    norm_name = attn_org_name.rpartition("/")[0]+f'.norm{attn_org_name.rpartition("/")[2][-1]}'
    norm = get_nested_attr(unet,norm_name,ignore_last=0)
    norm.weight = None
    norm.bias = None 


    

In [ ]:
if layer_norm_input and  kqv is False and ff_input is False : 
    # attn2 (only?)
    suffix = 'to_q'
    org_name_=None
    for org_name in org_names:
        if 'attn' in org_name and 'MatMul' in org_name:
            org_name_ = org_name
            break
    assert(org_name_ is not None)
    #attn_org_name = org_name_.rpartition("/")[0] 
    org_name = org_name_#attn_org_name+ '/' + suffix
    module = get_nested_attr(unet,org_name,ignore_last=1)
    p_weights = module.weight.data#.shape,
    if module.bias is not None:
        p_bias = module.bias.data
    else:
        p_bias = None
    assert(p_weights.numel() == ker.size)
    if p_bias is not None:
        assert(p_bias.numel() == bias.size)
    p_ker = acc_ker_to_pytorch_weight(ker)
    assert(p_weights.shape == p_ker.shape)
    module.weight.data =  p_ker
    if bias is not None:
        if p_bias is not None:
            module.bias.data =  torch.from_numpy(bias).reshape(p_bias.shape)
        else:
            module.bias =  nn.Parameter(torch.from_numpy(bias).reshape(-1))
    else:
        module.bias = None
    
    ## remove weight and bias from layer_normalization. 
    
    norm_name = '/'.join(org_name_.split("/")[:-3])+f'.norm{2}'
    norm = get_nested_attr(unet,norm_name,ignore_last=0)
    norm.weight = None
    norm.bias = None 

    

In [282]:
norm.weight.shape,norm.bias.shape   

(torch.Size([640]), torch.Size([640]))

In [253]:
layer_name = layers_names[ind]
layer_name

'unet_sim/conv_feature_splitter22_2'

In [283]:
#if ff_input and layer_norm_input :
    #suffix = 'to_q'
layer_name = layers_names[ind]
org_name_=None
for org_name in org_names:
    if 'proj' in org_name and 'MatMul' in org_name:
        org_name_ = org_name
        break
assert(org_name_ is not None)
org_name = org_name_

module = get_nested_attr(unet,org_name_,ignore_last=1)
p_weights = module.weight.data#.shape,
if module.bias is not None:
    p_bias = module.bias.data
else:
    p_bias = None
assert(p_weights.numel()/2 == ker.size)
if p_bias is not None:
    assert(p_bias.numel()/2 == bias.size)

if layer_name.endswith('_1'):
    right_part = True
elif layer_name.endswith('_2'):
    right_part = False
else:
    raise ValueError(f'layer name {layer_name} should end with _1 or _2')
center_channel = p_weights.shape[0]//2
p_ker = acc_ker_to_pytorch_weight(ker)
if right_part:
    module.weight.data[:center_channel] =  p_ker
    if p_bias is not None:
        module.bias.data[:center_channel] =  torch.from_numpy(bias).reshape(-1)
else:
    module.weight.data[center_channel:] =  p_ker
    if p_bias is not None:
        module.bias.data[center_channel:] =  torch.from_numpy(bias).reshape(-1)

norm_name = [on for on in org_names if 'LayerNormalization' in on]
if len(norm_name) == 0:
    raise ValueError('no norm name found')
norm_name = norm_name[0]
norm = get_nested_attr(unet,norm_name,ignore_last=1)
norm.weight = None
norm.bias = None 

    
    

In [ ]:
norm_name = '/'.join(org_name_.split("/")[:-3])+f'.norm{2}'
norm = get_nested_attr(unet,norm_name,ignore_last=0)
norm.weight = None
norm.bias = None 

In [278]:
import torch
import numpy as np

# Example: TensorFlow-style kernel (H, W, in_channels, out_channels)
tf_kernel = np.random.randn(3, 3, 320, 1280)  # Simulating a TF kernel

# Convert to PyTorch tensor
torch_kernel = torch.tensor(tf_kernel)

# Permute dimensions: (H, W, in_channels, out_channels) -> (out_channels, in_channels, H, W)
torch_kernel = torch_kernel.permute(3, 2, 0, 1)

print(torch_kernel.shape)  # Output: torch.Size([1280, 1280, 3, 3])


torch.Size([1280, 320, 3, 3])


In [275]:
p_ker.shape,ker.shape,p_weights.shape

(torch.Size([2560, 640]), (1, 1, 640, 2560), torch.Size([5120, 640]))

In [263]:
center_channel = p_weights.shape[0]//2
center_channel

2560

In [259]:
p_weights.numel()/2 , ker.size

(1638400.0, 1638400)

In [257]:
#org_names
#org_name,org_name_,
p_weights.shape, ker.shape
#p_bias.shape

(torch.Size([5120, 640]), (1, 1, 640, 2560))

In [248]:
module = get_nested_attr(unet,org_name_,ignore_last=1)
org_name_,module.full_name

('/up_blocks.2/attentions.0/transformer_blocks.0/ff/net.0/proj/MatMul',
 'up_blocks.2.attentions.0.transformer_blocks.0.ff.net.0.proj')

In [249]:
p_weights = module.weight.data
p_weights.shape

torch.Size([5120, 640])

In [ ]:
#attn_org_name = org_name_.rpartition("/")[0] 
attn_org_name.rpartition("/")[2][-1]

'1'

In [193]:
attn_org_name

'/mid_block/attentions.0/transformer_blocks.0/attn1'

In [194]:
norm_name = attn_org_name.rpartition("/")[0]+f'.norm{attn_org_name.rpartition("/")[2][-1]}'

In [196]:
get_nested_attr(unet,norm_name,ignore_last=0)
norm_name

'/mid_block/attentions.0/transformer_blocks.0.norm1'

In [181]:
module.weight.data.shape

torch.Size([1280, 1280])

In [182]:
module(torch.randn(1,1280))

tensor([[ 1.3892, -0.7074, -0.4748,  ...,  0.4830,  0.2412,  1.2872]],
       grad_fn=<AddmmBackward0>)

In [168]:
bias.shape

(1280,)

In [139]:
module = get_nested_attr(unet,org_name,ignore_last=0)
module.full_name

'mid_block.attentions.0.transformer_blocks.0.attn1.to_v'

In [131]:
orn_name_

'/mid_block/attentions.0/transformer_blocks.0/attn1/Reshape'

In [ ]:
org_name_=None
for org_name in org_names:
    if 'attn' in org_name:
        orn_name_ = org_name
assert(org_name_ is not None)
org_name = org_name_.rpartition("/")[0] + '/' + suffix

NameError: name 'org_name_' is not defined

In [ ]:
layer_norm_input and kqv

True

In [82]:
#p_weights.shape , ker.shape
p_bais.shape , bais.shape


(torch.Size([1280]), (1280,))

In [63]:
calc_snr(p_weights.cpu().numpy().transpose(),ker)

122.57843713777643

In [271]:
def acc_ker_to_pytorch_weight(ker):
    ker = ker.squeeze()
    if len(ker.shape) == 2:
        ker = torch.tensor(ker.transpose(1,0))
    elif len(ker.shape) == 4:
        ker = torch.tensor(ker.transpose(3,2,0,1))
    return ker

In [272]:
p_ker = acc_ker_to_pytorch_weight(ker)

In [273]:
ker.shape,p_ker.shape

((1, 1, 640, 2560), torch.Size([2560, 640]))

In [81]:
calc_snr(p_ker.cpu().numpy(),p_weights.cpu().numpy())

120.16079485226263

In [92]:
type(unet.down_blocks[0].attentions[0].transformer_blocks[0] )#/norm1/LayerNormalization

diffusers.models.attention.BasicTransformerBlock

In [ ]:
attn1 = unet.down_blocks[0].attentions[0].transformer_blocks[0].attn1

In [102]:
attn1.to_q.weight.shape , attn1.to_k.weight.shape , attn1.to_v.weight.shape


(torch.Size([320, 320]), torch.Size([320, 320]), torch.Size([320, 320]))

In [15]:
type(unet.down_blocks[0].attentions[0].transformer_blocks[0].ff.net[0])#[0])
#len(unet.down_blocks[0].attentions[0].transformer_blocks[0].ff.net)

diffusers.models.activations.GEGLU

In [41]:
gegelu = unet.down_blocks[0].attentions[0].transformer_blocks[0].ff.net[0]
gegelu

GEGLU(
  (proj): Linear(in_features=320, out_features=2560, bias=True)
)

In [44]:
#gegelu.proj,
type(gegelu.gelu)

method

In [45]:
import torch.nn.functional as F
F.gelu

<function torch._C._nn.gelu>

In [35]:
unet.up_blocks[3].attentions[0].transformer_blocks[0].ff.net[2].weight.shape

torch.Size([320, 1280])

In [78]:
type(unet.down_blocks[1])#.attentions[0]

diffusers.models.unets.unet_2d_blocks.CrossAttnDownBlock2D

In [80]:
len(unet.down_blocks[1].downsamplers)

1

In [83]:
unet.down_blocks[1].downsamplers

ModuleList(
  (0): Downsample2D(
    (conv): Conv2d(640, 640, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  )
)

In [16]:
ace_unet_path =  '/genai/users/ellaf/stable_diffusion/unet/shared_unet_demo/unet_sim_fp.har'

In [17]:
get_har_files(ace_unet_path)

['unet_sim.hn',
 'unet_sim.alls',
 'unet_sim.native.hn',
 'unet_sim.npz',
 'unet_sim.fpo.npz',
 'unet_sim.stats.npz',
 'unet_sim.original_model_meta.json',
 'unet_sim.metadata.json']

In [ ]:
params = get_params_from_har(ace_unet_path,params_name='unet_sim.npz')

all_names=['unet_sim.hn', 'unet_sim.alls', 'unet_sim.native.hn', 'unet_sim.npz', 'unet_sim.fpo.npz', 'unet_sim.stats.npz', 'unet_sim.original_model_meta.json', 'unet_sim.metadata.json']
loading  params_name='unet_sim.npz' ...


In [16]:
params.keys()

dict_keys(['unet_sim/conv1/padding_const_value:0', 'unet_sim/conv1/kernel:0', 'unet_sim/conv1/bias:0', 'unet_sim/layer_normalization103/epsilon:0', 'unet_sim/normalization103/kernel:0', 'unet_sim/normalization103/bias:0', 'unet_sim/layer_normalization1/epsilon:0', 'unet_sim/normalization1/kernel:0', 'unet_sim/normalization1/bias:0', 'unet_sim/fc1/kernel:0', 'unet_sim/fc1/bias:0', 'unet_sim/conv12/padding_const_value:0', 'unet_sim/conv12/kernel:0', 'unet_sim/conv12/bias:0', 'unet_sim/mul_and_add1/kernel:0', 'unet_sim/mul_and_add1/bias:0', 'unet_sim/conv23/padding_const_value:0', 'unet_sim/conv23/kernel:0', 'unet_sim/conv23/bias:0', 'unet_sim/conv28/padding_const_value:0', 'unet_sim/conv28/kernel:0', 'unet_sim/conv28/bias:0', 'unet_sim/mul_and_add9/kernel:0', 'unet_sim/mul_and_add9/bias:0', 'unet_sim/conv29/padding_const_value:0', 'unet_sim/conv29/kernel:0', 'unet_sim/conv29/bias:0', 'unet_sim/conv30/padding_const_value:0', 'unet_sim/conv30/kernel:0', 'unet_sim/conv30/bias:0', 'unet_sim/

In [20]:
hn = get_params_from_har(ace_unet_path,params_name='unet_sim.hn')

all_names=['unet_sim.hn', 'unet_sim.alls', 'unet_sim.native.hn', 'unet_sim.npz', 'unet_sim.fpo.npz', 'unet_sim.stats.npz', 'unet_sim.original_model_meta.json', 'unet_sim.metadata.json']
loading  params_name='unet_sim.hn' ...


In [21]:
#[ki for ki in hn['layers'].keys() if 'input' in ki]
#[hn['layers'][ki] for ki in hn['layers'].keys() if 'input' in ki]

In [22]:
k = list(hn['layers'].keys())

In [26]:
k[:10]

['unet_sim/input_layer1',
 'unet_sim/conv1',
 'unet_sim/input_layer2',
 'unet_sim/fc1',
 'unet_sim/fc2',
 'unet_sim/fc10',
 'unet_sim/fc11',
 'unet_sim/fc12',
 'unet_sim/fc13',
 'unet_sim/fc14']

In [65]:
ind = 60
k[ind], hn['layers'][k[ind]]['original_names'],hn['layers'][k[ind]]['type']

('unet_sim/layer_normalization1',
 ['/down_blocks.0/resnets.0/norm1/Reshape',
  '/down_blocks.0/resnets.0/norm1/InstanceNormalization',
  '/down_blocks.0/resnets.0/norm1/Reshape_1'],
 'layer_normalization')

In [66]:
hn['layers'][k[ind]]['original_names']

['/down_blocks.0/resnets.0/norm1/Reshape',
 '/down_blocks.0/resnets.0/norm1/InstanceNormalization',
 '/down_blocks.0/resnets.0/norm1/Reshape_1']

In [67]:
#ind =75
#ind =13
ind =60
obj_pa= get_nested_attr(unet,hn['layers'][k[ind]]['original_names'][0],2)
obj= get_nested_attr(unet,hn['layers'][k[ind]]['original_names'][0])
hn['layers'][k[ind]]['original_names'][0], obj_pa.full_name, obj.full_name,  obj

('/down_blocks.0/resnets.0/norm1/Reshape',
 'down_blocks.0.resnets.0',
 'down_blocks.0.resnets.0.norm1',
 GroupNorm(32, 320, eps=1e-05, affine=True))

In [68]:
type(obj_pa),type(obj)

(diffusers.models.resnet.ResnetBlock2D,
 torch.nn.modules.normalization.GroupNorm)

In [69]:
k[ind]

'unet_sim/layer_normalization1'

In [70]:
kp = [ki for ki in params.keys() if k[ind]+'/' in ki]
kp

['unet_sim/layer_normalization1/epsilon:0']

In [51]:
kernel_key = [ki for ki in kp if 'kernel' in ki][0]
bias_key = [ki for ki in kp if 'bias' in ki][0]
kernel_key, bias_key

('unet_sim/conv12/kernel:0', 'unet_sim/conv12/bias:0')

In [52]:
wa = params[kernel_key] 
ba = params[bias_key]
wa.shape,ba.shape

((1, 1, 768, 320), (320,))

In [56]:
obj.bias

In [71]:
obj.weight.data.numpy().shape,#obj.bias.data.numpy().shape

((320,),)

In [72]:
obj.weight.data.numpy().shape

(320,)

torch.nn.modules.linear.Linear

In [61]:
dw = wa[0,0] - obj.weight.data.numpy().transpose(1,0)
dw.shape

(768, 320)

In [62]:
calc_snr( wa[0,0]  , obj.weight.data.numpy().transpose(1,0))

131.95639515419396

In [33]:
calc_snr(ba, obj.bias.data.numpy())

131.4684181282942

In [130]:
obj= get_nested_attr(unet,hn['layers'][k[ind]]['original_names'][0],ignore_last=2).norm2
obj.weight.shape,obj.bias.shape

(torch.Size([320]), torch.Size([320]))

In [141]:
obj.weight.detach().numpy().shape,w.shape

((320,), (1, 1, 320, 1))

In [144]:
d = obj.weight.detach().numpy()-w[0,0,:,0]
d.shape

(320,)

In [145]:
d

array([-1.37381464e-01, -3.92369330e-02, -4.58287299e-02, -1.92068964e-01,
       -9.70982611e-02, -2.11478144e-01, -1.56180292e-01, -9.97838080e-02,
       -1.75833613e-01, -1.04422480e-01, -7.48814642e-02, -1.50320917e-01,
       -3.97252142e-02, -5.65709174e-02, -8.02525580e-02, -1.38113886e-01,
       -9.24595892e-02, -3.58189642e-02, -7.82994330e-02, -7.98693299e-03,
       -5.68150580e-02, -9.63658392e-02, -1.05887324e-01, -1.27127558e-01,
       -1.40799433e-01, -8.92857611e-02, -1.03445917e-01, -8.90416205e-02,
       -7.58580267e-02, -8.36705267e-02, -1.09793574e-01, -1.95609003e-01,
       -8.83091986e-02, -1.06863886e-01, -1.11014277e-01, -9.97838080e-02,
       -9.63658392e-02, -5.43736517e-02, -9.46568549e-02, -9.17271674e-02,
       -1.52396113e-01, -5.31529486e-02, -1.24930292e-01, -1.30301386e-01,
       -8.19615424e-02, -3.67955267e-02, -2.23912299e-02, -1.10037714e-01,
       -1.22977167e-01, -7.29283392e-02,  2.37513483e-02, -5.07115424e-02,
       -4.97349799e-02, -

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained("SG161222/Realistic_Vision_V4.0_noVAE")

In [92]:
#!/usr/bin/env python
from typing import Optional, Union

import torch
from diffusers import AutoencoderKL, DDIMScheduler, StableDiffusionPipeline

base_model_path = "SG161222/Realistic_Vision_V4.0_noVAE"
orig_base_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"
vae_model_path = "stabilityai/sd-vae-ft-mse"
device = "cuda"
dtype = torch.float32

scheduler = DDIMScheduler.from_pretrained(base_model_path, subfolder="scheduler")
vae = AutoencoderKL.from_pretrained(vae_model_path, torch_dtype=dtype)
pipe = StableDiffusionPipeline.from_pretrained(
    orig_base_model_path,
    torch_dtype=dtype,
    scheduler=scheduler,
    vae=vae,
    # feature_extractor=AutoFeatureExtractor.from_pretrained(
    #     orig_base_model_path, subfolder="feature_extractor", torch_dtype=dtype
    # ),
    safety_checker=None,
)

unet = pipe.unet
add_full_name_to_module(unet)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [93]:
import onnx

In [94]:
onnx_path = '/genai/users/shacharg/stable_diffusion/2025-02-06/unet_sim.onnx'

In [95]:
onnx_path = '/genai/users/shacharg/stable_diffusion/2025-02-06/unet_sim.onnx'
onnx_model = onnx.load(onnx_path)

In [96]:
type(onnx_model)

onnx.onnx_ml_pb2.ModelProto

In [97]:
#/up_blocks.3/resnets.0/time_emb_proj/#''

In [98]:
onnx_model.graph.node[0].input


['/time_proj/Concat_1_output_0', 'time_embedding.linear_1.weight', 'time_embedding.linear_1.bias']

In [99]:
import numpy as np
for initializer in onnx_model.graph.initializer:
    if "time_emb_proj" in initializer.name:
        weight_array = np.frombuffer(initializer.raw_data, dtype=np.float32).reshape(initializer.dims)
        print(f"Found weight for {initializer.name} with shape {initializer.dims}")
        break

Found weight for down_blocks.0.resnets.0.time_emb_proj.weight with shape [320, 1280]


In [100]:
w= unet.down_blocks[0].resnets[0].time_emb_proj.weight.data.numpy()

In [101]:
weight_array.shape

(320, 1280)

In [102]:
calc_snr(weight_array,w)

118.30428327609846

In [91]:
wa.shape , wa.transpose(1,0).shape,w.shape

((1280, 320), (320, 1280), (320, 1280))

In [108]:
#wa.transpose(1,0).shape
calc_snr(w,wa.transpose(1,0))

118.30428327609846

In [53]:
weight_array-w

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)